# Projeto wellbe_rpa_pipeline

## Introducao
Este notebook apresenta uma solucao ETL automatizada com RPA para coleta de filmes no portal [RPA Challenge](https://rpachallenge.com/), com foco em organizacao de codigo, robustez de automacao e padrao de entrega profissional.

## Objetivo
Construir e demonstrar um pipeline completo de **Extract, Transform, Load (ETL)** que:

1. Extrai filmes da aba Movie Search para a busca `Avengers`.
2. Transforma os dados para consistencia e deduplicacao.
3. Carrega os resultados em MySQL com controle de duplicidade.
4. Gera outputs rastreaveis (CSV, JSON e dump SQL).

## Arquitetura da Solucao

Fluxo logico do pipeline:

```mermaid
flowchart LR
    A[Selenium RPA\nRPA Challenge - Movie Search] --> B[Extract\nColeta nome e descricao completa]
    B --> C[Transform\nLimpeza, padronizacao, deduplicacao]
    C --> D[Load\nMySQL upsert]
    C --> E[Outputs\nCSV e JSON]
    C --> F[Dump SQL\nreexecucao e auditoria]
```

### Estrutura de Projeto

- `wellbe_rpa_pipeline.ipynb`: relatorio tecnico e execucao guiada.
- `src/scraper.py`: automacao Selenium com waits explicitos.
- `src/database.py`: conexao MySQL, schema e upsert.
- `src/utils.py`: funcoes utilitarias de transformacao e output.
- `src/config.py`: configuracoes via variaveis de ambiente.
- `src/pipeline.py`: orquestracao da execucao fim a fim.
- `sql/create_movies_table.sql`: script de criacao da tabela.
- `outputs/`: arquivos finais CSV e JSON.
- `data/`: camada de persistencia intermediaria (raw).

## Tecnologias Utilizadas

- **Python 3**: linguagem principal do pipeline.
- **Selenium**: automacao RPA para navegacao e scraping.
- **Pandas / Numpy**: transformacao de dados tabulares.
- **MySQL**: persistencia relacional com upsert.
- **Jupyter Notebook**: documentacao tecnica e reproducibilidade.

## Etapas do Pipeline (ETL)

- **Extract**: navega no site, acessa Movie Search, busca `Avengers` e captura nome + descricao completa via `card-reveal`.
- **Transform**: remove vazios, normaliza strings, elimina duplicados e organiza dataset.
- **Load**: garante schema da tabela `movies`, realiza insercao com controle de duplicidade e gera dump SQL versionado.

In [1]:
# Setup inicial
from pathlib import Path
import logging
import pandas as pd
import numpy as np

from src.config import load_config
from src.scraper import RPAMovieScraper
from src.database import MySQLMovieRepository, build_dump_sql
from src.utils import configure_logging, ensure_directories, normalize_movies_df, now_stamp, save_dataframe_outputs

configure_logging(logging.INFO)
config = load_config()
ensure_directories([config.data_dir, config.output_dir, config.sql_dir])

print("Configuracao carregada.")
print(f"Base URL: {config.base_url}")
print(f"Consulta padrao: {config.default_query}")
print(f"Headless: {config.headless}")

Configuracao carregada.
Base URL: https://rpachallenge.com/
Consulta padrao: Avengers
Headless: True


## Execucao Passo a Passo

### 1. Extract (RPA + Selenium)
Nesta etapa, o robô:

1. Abre a home page.
2. Clica em **Movie Search**.
3. Busca por `Avengers`.
4. Para cada card, expande `card-reveal` e captura a descricao completa.

In [2]:
# EXTRACT
raw_movies = []

try:
    with RPAMovieScraper(
        base_url=config.base_url,
        movie_search_path=config.movie_search_path,
        timeout_seconds=config.timeout_seconds,
        headless=config.headless,
    ) as scraper:
        raw_movies = scraper.run(query=config.default_query)

    print(f"Total de filmes extraidos: {len(raw_movies)}")
except Exception as exc:
    print(f"Falha na etapa EXTRACT: {exc}")
    raise

raw_df = pd.DataFrame(raw_movies)
raw_path = config.data_dir / "movies_raw.csv"
raw_df.to_csv(raw_path, index=False, encoding="utf-8")

display(raw_df.head())
print(f"Arquivo raw salvo em: {raw_path}")

2026-04-17 20:41:41,520 | INFO | src.scraper | WebDriver iniciado
2026-04-17 20:41:45,898 | INFO | src.scraper | Cards encontrados: 3
2026-04-17 20:41:48,622 | INFO | src.scraper | WebDriver finalizado


Total de filmes extraidos: 3


,name,description
0,The Avengers,
1,Avengers: Infinity W,
2,Avengers: Endgame,


Arquivo raw salvo em: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\data\movies_raw.csv


### 2. Transform

Regras aplicadas:

- Trim de espacos
- Exclusao de nulos/vazios
- Remocao de duplicados por nome do filme
- Ordenacao alfabetica para facilitar auditoria

In [3]:
# TRANSFORM
transformed_df = normalize_movies_df(raw_df)

stamp = now_stamp()
outputs = save_dataframe_outputs(
    transformed_df,
    output_dir=config.output_dir,
    base_name=f"movies_avengers_{stamp}",
)

print(f"Linhas antes: {len(raw_df)}")
print(f"Linhas depois: {len(transformed_df)}")
print(f"CSV: {outputs['csv']}")
print(f"JSON: {outputs['json']}")

display(transformed_df.head(10))

Linhas antes: 3
Linhas depois: 0
CSV: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260417_204220.csv
JSON: C:\Users\d4rkm0d3\Desktop\Camille\wellbe_rpa_pipeline\outputs\movies_avengers_20260417_204220.json


,name,description


### 3. Load (MySQL)

Nesta etapa a tabela `movies` e validada/criada e os dados sao persistidos com estrategia de `upsert` para evitar duplicidades em reexecucoes.

In [ ]:
# LOAD
inserted_rows = 0
load_error = None

try:
    with MySQLMovieRepository(
        host=config.mysql_host,
        port=config.mysql_port,
        database=config.mysql_database,
        user=config.mysql_user,
        password=config.mysql_password,
    ) as repo:
        repo.ensure_schema()
        inserted_rows = repo.upsert_movies(transformed_df.to_dict(orient="records"))
    print(f"Carga concluida. Linhas processadas: {inserted_rows}")
except Exception as exc:
    load_error = str(exc)
    print("Carga no MySQL nao executada com sucesso nesta maquina.")
    print(f"Motivo: {load_error}")
    print("Os arquivos de output e o dump SQL continuam disponiveis para validacao tecnica.")

### 4. Geracao do Dump SQL

Gera um arquivo SQL versionado contendo:

- DDL da tabela `movies`
- Bloco de insercao dos registros transformados
- Estrategia `ON DUPLICATE KEY UPDATE`

In [ ]:
dump_path = build_dump_sql(
    transformed_df.to_dict(orient="records"),
    output_file=config.sql_dir / f"movies_dump_{stamp}.sql",
)

print(f"Dump SQL gerado em: {dump_path}")

## Resultados

Resumo executivo da execucao do pipeline.

In [ ]:
results_df = pd.DataFrame(
    [
        {"metrica": "Registros extraidos (raw)", "valor": len(raw_df)},
        {"metrica": "Registros transformados", "valor": len(transformed_df)},
        {"metrica": "Linhas processadas no load", "valor": inserted_rows},
        {"metrica": "Falha no load", "valor": "Sim" if load_error else "Nao"},
    ]
)

display(results_df)

print("\nPrincipais artefatos gerados:")
print(f"- Raw CSV: {raw_path}")
print(f"- Output CSV: {outputs['csv']}")
print(f"- Output JSON: {outputs['json']}")
print(f"- Dump SQL: {dump_path}")

## Conclusao

A solucao implementa um pipeline ETL de ponta a ponta com separacao de responsabilidades, automacao robusta e outputs auditaveis.

Pontos de destaque:

- Extracao baseada em seletores estaveis e waits explicitos.
- Captura da descricao completa via interacao controlada com `card-reveal`.
- Transformacao idempotente com deduplicacao.
- Carga preparada para reexecucao com `upsert`.
- Entrega de artefatos tecnicos (CSV, JSON e dump SQL) para rastreabilidade.

Com pequenos ajustes de infraestrutura (segredos, observabilidade e orquestracao), o projeto esta pronto para evoluir para ambiente produtivo.